# Dataset Viewer Notebook

This notebook recreates the functionality of `src/tools/viewer.py` using PyQt5 and OpenCV, and displays dataset images, masks, and mask pixel statistics in a notebook-friendly form.

In [ ]:
# 1. Import required libraries

import os
import re
import cv2
import numpy as np

from PyQt5.QtWidgets import QApplication, QWidget, QLabel, QPushButton, QHBoxLayout, QVBoxLayout
from PyQt5.QtGui import QPixmap, QImage
from PyQt5.QtCore import Qt

In [ ]:
# 2. Define project root and dataset paths

base_dir = os.path.abspath(os.path.join(os.path.dirname(__file__), '..'))
image_dir = os.path.join(base_dir, 'output/v2/images')
mask_dir = os.path.join(base_dir, 'output/v2/masks')

print('Image directory:', image_dir)
print('Mask directory:', mask_dir)

In [ ]:
# 3. Create filename sorting helper

def extract_num(filename):
    numbers = re.findall(r"\d+", filename)
    return int(numbers[-1]) if numbers else -1

try:
    image_files = sorted(os.listdir(image_dir), key=extract_num)
except FileNotFoundError:
    image_files = []

print('Total images:', len(image_files))

In [ ]:
# 4. Build Viewer UI with PyQt5 widgets

class Viewer(QWidget):
    def __init__(self, image_dir, mask_dir, image_files):
        super().__init__()
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.image_files = image_files
        self.index = 0

        self.init_ui()
        self.load_current()

    def init_ui(self):
        self.setWindowTitle('Dataset Viewer')
        self.resize(1400, 800)

        self.img_label = QLabel()
        self.mask_label = QLabel()
        self.stats_label = QLabel()

        self.img_label.setAlignment(Qt.AlignCenter)
        self.mask_label.setAlignment(Qt.AlignCenter)
        self.stats_label.setAlignment(Qt.AlignLeft | Qt.AlignTop)
        self.stats_label.setWordWrap(True)

        self.prev_btn = QPushButton('Previous')
        self.next_btn = QPushButton('Next')

        self.prev_btn.clicked.connect(self.prev)
        self.next_btn.clicked.connect(self.next)

        top_layout = QHBoxLayout()
        top_layout.addWidget(self.img_label)
        top_layout.addWidget(self.mask_label)

        btn_layout = QHBoxLayout()
        btn_layout.addStretch()
        btn_layout.addWidget(self.prev_btn)
        btn_layout.addWidget(self.next_btn)
        btn_layout.addStretch()

        main_layout = QVBoxLayout()
        main_layout.addLayout(top_layout)
        main_layout.addWidget(self.stats_label)
        main_layout.addLayout(btn_layout)

        self.setLayout(main_layout)

    def load_current(self):
        if not self.image_files:
            self.stats_label.setText('No images found.')
            return

        image_name = self.image_files[self.index]
        img_path = os.path.join(self.image_dir, image_name)
        mask_name = os.path.splitext(image_name)[0] + '.png'
        mask_path = os.path.join(self.mask_dir, mask_name)

        img = cv2.imread(img_path)
        mask = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)

        if img is None or mask is None:
            self.stats_label.setText(f'Failed to load image or mask: {image_name}')
            return

        if mask.ndim == 3:
            mask = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)

        unique, counts = np.unique(mask, return_counts=True)
        total_pixels = mask.size

        stats_lines = [
            f'Index: {self.index + 1}/{len(self.image_files)}',
            f'Image: {image_name}',
            f'Image shape: {img.shape}',
            f'Mask shape: {mask.shape}',
            'Mask values:'
        ]
        for value, count in zip(unique, counts):
            stats_lines.append(f'  value={value}: {count} pixels, ratio={count / total_pixels:.6f}')

        invalid_values = [v for v in unique if v not in (0, 1, 2)]
        if invalid_values:
            stats_lines.append(f'Invalid values: {invalid_values}')
        else:
            stats_lines.append('Invalid values: none')

        self.stats_label.setText('\n'.join(stats_lines))

        mask_color = np.zeros_like(img)
        mask_color[mask == 0] = [0, 0, 0]
        mask_color[mask == 1] = [0, 255, 0]
        mask_color[mask == 2] = [0, 0, 255]

        self.show_img(self.img_label, img)
        self.show_img(self.mask_label, mask_color)

    def show_img(self, label, img):
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w, c = img_rgb.shape
        qimg = QImage(img_rgb.data, w, h, c * w, QImage.Format_RGB888)
        pix = QPixmap.fromImage(qimg)
        pix = pix.scaled(650, 700, Qt.KeepAspectRatio)
        label.setPixmap(pix)

    def prev(self):
        self.index = max(0, self.index - 1)
        self.load_current()

    def next(self):
        self.index = min(len(self.image_files) - 1, self.index + 1)
        self.load_current()

    def keyPressEvent(self, event):
        if event.key() == Qt.Key_A:
            self.prev()
        elif event.key() == Qt.Key_D:
            self.next()

In [ ]:
# 5. Launch the viewer

app = QApplication([])
viewer = Viewer(image_dir, mask_dir, image_files)
viewer.show()
app.exec_()